In [1]:
# Parameters — change these and re-run
DATE = "2021-02-21"
PORTFOLIO = "60/40 Multi-Asset"
ESTIMATION_WINDOW = 252
REBALANCE_TOLERANCE = 0.05


In [2]:
import sys
from pathlib import Path

# Locate the project root from either the project root or this notebook's directory.
current_directory = Path.cwd().resolve()
project_root = next(
    path for path in (current_directory, *current_directory.parents)
    if (path / "pyproject.toml").exists()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from IPython.display import Markdown, display
from src.display.rebalance_review_renderer import render_rebalance_review_markdown
from src.display.risk_drift_renderer import render_risk_drift_markdown
from src.engine.attribution_engine import AttributionEngine
from src.engine.rebalance_trigger_engine import RebalanceTriggerEngine
from src.engine.risk_drift_engine import RiskDriftEngine
from src.infrastructure.in_memory_portfolios import InMemoryPortfolioRepository
from src.infrastructure.yfinance_returns import YFinanceReturnsProvider


In [3]:
# Composition root — risk measurement feeds a tolerance-band rebalance review.
portfolios = InMemoryPortfolioRepository()
returns_provider = YFinanceReturnsProvider()
attribution_engine = AttributionEngine(portfolios, returns_provider)
risk_drift_engine = RiskDriftEngine()
rebalance_trigger_engine = RebalanceTriggerEngine()

attribution = attribution_engine.attribute(
    portfolio_name=PORTFOLIO,
    date=DATE,
    estimation_window=ESTIMATION_WINDOW,
)
portfolio = portfolios.get(PORTFOLIO)
risk_drift = risk_drift_engine.calculate(attribution, portfolio)
rebalance_trigger = rebalance_trigger_engine.evaluate(
    risk_drift,
    tolerance=REBALANCE_TOLERANCE,
)

display(Markdown(render_risk_drift_markdown(
    risk_drift,
    portfolio_name=PORTFOLIO,
    date=DATE,
)))

display(Markdown(render_rebalance_review_markdown(
    rebalance_trigger,
    portfolio_name=PORTFOLIO,
    date=DATE,
)))


**RISK CONTRIBUTION DRIFT**<br><br>**60/40 Multi-Asset**<br>As of 21 February 2021<br><br>

| Asset | Weight | Target risk contribution | Current risk contribution | Signed drift | Absolute drift |
|---|---:|---:|---:|---:|---:|
| IEF | 25% | 20% | -3% | -23.35 pp | 23.35 pp |
| SPY | 40% | 45% | 67% | +22.27 pp | 22.27 pp |
| EFA | 20% | 20% | 31% | +10.65 pp | 10.65 pp |
| GLD | 15% | 15% | 5% | -9.57 pp | 9.57 pp |



**RISK DRIFT — REBALANCE REVIEW**<br><br>**60/40 Multi-Asset**<br>As of 21 February 2021<br><br>

Tolerance band: ±5.00 percentage points<br><br>

**REVIEW REBALANCE**<br><br>

| Asset | Target | Current | Drift | Band | Status | Suggested direction |
|---|---:|---:|---:|---:|---|---|
| IEF | 20% | -3% | -23.35 pp | ±5.00 pp | Outside | Consider increasing |
| SPY | 45% | 67% | +22.27 pp | ±5.00 pp | Outside | Consider reducing |
| EFA | 20% | 31% | +10.65 pp | ±5.00 pp | Outside | Consider reducing |
| GLD | 15% | 5% | -9.57 pp | ±5.00 pp | Outside | Consider increasing |

The trigger identifies risk-contribution drift outside the configured tolerance band. It does not determine trade size, execution, cost, liquidity, or whether the PM should rebalance.



The first table presents current risk-contribution drift versus the mandate. The second applies a configurable tolerance band and identifies assets that warrant PM review. Directional suggestions are candidate review directions only: this notebook does not determine trade size, execution, transaction costs, liquidity, or whether the PM should rebalance.